# 11 · Faster R-CNN and an explicit transfer-learning loop

**Level:** Advanced · **Time:** 45–60 minutes plus model download/compute.
**Prerequisites:** notebooks 06–09; basic Python functions, tensors, and gradients;
compatible PyTorch/TorchVision packages from the project requirements.

**You will:** build a TorchVision detection Dataset, replace a classifier head,
inspect loss components, perform two real optimizer steps, and run inference.

**Demo story:** YOLO's training call packages the loop for us. Here we expose the
data contract and gradient update using a two-stage Faster R-CNN model.
Two batches demonstrate the mechanics; they are far too few to claim convergence.

The normal path downloads pretrained weights once. With `COURSE_OFFLINE=1`,
construction uses random weights, downloads nothing, and skips the expensive
training/inference stages. Set `COURSE_FRCNN_OFFLINE_TRAIN=1` explicitly to test
two updates on the random model offline; those are architecture checks, not
pretrained transfer learning.


In [ ]:
from pathlib import Path
import sys
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "course_utils.py").exists()), None)
if ROOT is None:
    raise FileNotFoundError("Open Jupyter from the ObjectDetection project folder (see README.md).")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from course_utils import seed_everything, show_boxes
seed_everything(42)
print("Project:", ROOT)


In [ ]:
import os, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
import torch
import torchvision
from torch.utils.data import Dataset, DataLoader
from torchvision.transforms.functional import pil_to_tensor
from course_utils import ensure_synthetic_dataset, read_yolo_labels

# Bootstrap precedes the torch import; seed torch now as well as NumPy/Python.
seed_everything(42)

OFFLINE = os.environ.get("COURSE_OFFLINE", "0") == "1"
OFFLINE_TRAIN = os.environ.get("COURSE_FRCNN_OFFLINE_TRAIN", "0") == "1"
USE_PRETRAINED = not OFFLINE
RUN_OPTIMIZER_DEMO = not OFFLINE or OFFLINE_TRAIN
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DATA_DIR = ensure_synthetic_dataset().parent
print("PyTorch:", torch.__version__, "TorchVision:", torchvision.__version__, "device:", DEVICE)


## 1. Understand the two-stage pipeline

`RGB image → backbone/FPN features → region proposals → ROI features → class + box`

The **region proposal network (RPN)** proposes candidate regions. The **ROI head**
classifies those regions and refines their coordinates. Training returns a
dictionary of classification, regression, objectness, and proposal-regression
losses. Evaluation returns boxes, labels, and scores.

Unlike our zero-based YOLO labels, TorchVision reserves **0 for background**.
Our rectangle, circle, and triangle labels become **1, 2, and 3**. The new head
therefore needs **4 classes**, including background.


In [ ]:
class ShapesDetectionDataset(Dataset):
    def __init__(self, split="train"):
        self.split = split
        self.image_paths = sorted((DATA_DIR / "images" / split).glob("*.png"))

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, index):
        path = self.image_paths[index]
        image = Image.open(path).convert("RGB")
        label_path = DATA_DIR / "labels" / self.split / (path.stem + ".txt")
        xyxy, class_ids = read_yolo_labels(label_path, *image.size)
        boxes = torch.as_tensor(xyxy, dtype=torch.float32).reshape(-1, 4)
        labels = torch.as_tensor(class_ids, dtype=torch.int64).reshape(-1) + 1
        area = (boxes[:, 2] - boxes[:, 0]) * (boxes[:, 3] - boxes[:, 1])
        target = {
            "boxes": boxes, "labels": labels,
            "image_id": torch.tensor(index, dtype=torch.int64),
            "area": area, "iscrowd": torch.zeros(len(boxes), dtype=torch.int64),
        }
        # Detector expects float CHW images in [0, 1]; it normalizes internally.
        tensor = pil_to_tensor(image).to(dtype=torch.float32) / 255.0
        return tensor, target

def detection_collate(batch):
    # Variable image sizes / object counts cannot be stacked as fixed targets.
    images, targets = zip(*batch)
    return list(images), list(targets)

train_dataset = ShapesDetectionDataset("train")
val_dataset = ShapesDetectionDataset("val")
loader = DataLoader(train_dataset, batch_size=2, shuffle=True, num_workers=0,
                    collate_fn=detection_collate,
                    generator=torch.Generator().manual_seed(42))
sample_image, sample_target = train_dataset[1]  # Index 0 is a deliberate negative example.
print("Image:", sample_image.shape, sample_image.dtype,
      "range:", (sample_image.min().item(), sample_image.max().item()))
print({k: (tuple(v.shape), str(v.dtype)) for k, v in sample_target.items()})
assert sample_target["boxes"].dtype == torch.float32
assert sample_target["labels"].dtype == torch.int64
assert torch.all((sample_target["labels"] >= 1) & (sample_target["labels"] <= 3))
show_boxes(sample_image.permute(1, 2, 0).numpy(), sample_target["boxes"].numpy(),
           (sample_target["labels"] - 1).numpy(), title="Dataset contract: displayed labels converted back to 0–2")
plt.show()


## 2. Replace the head and freeze the feature extractor

Use the MobileNetV3/FPN variant to keep the exercise relatively small. We reduce
the transform resolution to 256 and bound proposal counts for a classroom CPU
demo; this changes the compute/accuracy trade-off from the pretrained defaults.

The new `FastRCNNPredictor` learns our 4 outputs. We freeze the backbone to save
compute, then train the RPN/ROI parameters. In training mode we also put the
backbone in evaluation mode so its normalization behavior stays fixed. Batch
size 2 and this explicit freeze avoid the single-image BatchNorm pitfall.


In [ ]:
from torchvision.models.detection import (
    fasterrcnn_mobilenet_v3_large_320_fpn,
    FasterRCNN_MobileNet_V3_Large_320_FPN_Weights,
)
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor

weights = FasterRCNN_MobileNet_V3_Large_320_FPN_Weights.DEFAULT if USE_PRETRAINED else None
detector = fasterrcnn_mobilenet_v3_large_320_fpn(
    weights=weights,
    weights_backbone=None,  # Important: no implicit ImageNet download in random/offline mode.
    min_size=256, max_size=256,
    rpn_pre_nms_top_n_train=300, rpn_post_nms_top_n_train=150,
    rpn_pre_nms_top_n_test=150, rpn_post_nms_top_n_test=100,
    box_detections_per_img=30,
)
in_features = detector.roi_heads.box_predictor.cls_score.in_features
detector.roi_heads.box_predictor = FastRCNNPredictor(in_features, num_classes=4)
for parameter in detector.backbone.parameters():
    parameter.requires_grad = False
detector = detector.to(DEVICE)
trainable = [p for p in detector.parameters() if p.requires_grad]
print("Initialization:", "COCO-pretrained + new class head" if USE_PRETRAINED else "random/offline architecture")
print("Trainable parameters:", f"{sum(p.numel() for p in trainable):,}")
print("All parameters:", f"{sum(p.numel() for p in detector.parameters()):,}")
print(detector.roi_heads.box_predictor)


## 3. Make two real optimizer updates

The sequence is `zero_grad → forward → sum losses → backward → optimizer.step`.
The forward pass needs **images and targets** during training. A non-finite
loss is an error worth stopping for, not a number to hide.

**Presenter prompt:** Predict which head parameters should change after one
update. The code measures the classifier weight change to verify that an update
happened. A finite loss or a changed weight does not prove detection quality.


In [ ]:
loss_rows = []
if not RUN_OPTIMIZER_DEMO:
    print("Offline mode: model architecture built; optimizer/inference demo skipped.")
else:
    optimizer = torch.optim.SGD(trainable, lr=0.005, momentum=0.9, weight_decay=0.0005)
    detector.train()
    detector.backbone.eval()  # Freeze normalization statistics as well as gradients.
    initial_head = detector.roi_heads.box_predictor.cls_score.weight.detach().clone()
    started = time.perf_counter()
    for step, (images, targets) in enumerate(loader):
        if step >= 2:
            break
        images = [im.to(DEVICE) for im in images]
        targets = [{k: v.to(DEVICE) for k, v in target.items()} for target in targets]
        optimizer.zero_grad(set_to_none=True)
        losses = detector(images, targets)
        total_loss = sum(losses.values())
        if not torch.isfinite(total_loss):
            raise RuntimeError(f"Non-finite loss: {losses}")
        total_loss.backward()
        optimizer.step()
        row = {"step": step + 1, "total_loss": total_loss.detach().item(),
               **{k: v.detach().item() for k, v in losses.items()}}
        loss_rows.append(row)
        print(row)
    weight_change = (detector.roi_heads.box_predictor.cls_score.weight.detach() - initial_head).abs().max().item()
    assert len(loss_rows) == 2 and weight_change > 0
    print(f"Two updates took {time.perf_counter() - started:.1f}s; max head weight change={weight_change:.6g}")


In [ ]:
if loss_rows:
    loss_history = pd.DataFrame(loss_rows)
    display(loss_history.round(4))
    loss_history.set_index("step").drop(columns="total_loss").plot.bar(
        figsize=(9, 4), title="Measured loss terms for two different batches", rot=0)
    plt.ylabel("Loss"); plt.tight_layout(); plt.show()
else:
    print("Loss chart appears after an optimizer demo. No synthetic losses are shown.")


## 4. Switch to evaluation and inspect real predictions

Call `eval()` and use `torch.inference_mode()`. During evaluation the model needs
only images. Targets stay aside for scoring. Results use pixel `xyxy` boxes,
integer labels, and confidence scores. Filter all three arrays using the same
mask, then subtract 1 only when converting back to the course's class IDs.

Expect noisy detections after two updates, and especially with random offline
initialization. This cell demonstrates the output contract, not a finished model.


In [ ]:
if RUN_OPTIMIZER_DEMO:
    detector.eval()
    val_image, val_target = val_dataset[1]
    with torch.inference_mode():
        prediction = detector([val_image.to(DEVICE)])[0]
    prediction = {key: value.detach().cpu() for key, value in prediction.items()}
    CONFIDENCE = 0.30
    keep = prediction["scores"] >= CONFIDENCE
    fig, axes = plt.subplots(1, 2, figsize=(11, 4))
    rgb = val_image.permute(1, 2, 0).numpy()
    show_boxes(rgb, val_target["boxes"].numpy(), (val_target["labels"] - 1).numpy(),
               ax=axes[0], title="Validation ground truth")
    show_boxes(rgb, prediction["boxes"][keep].numpy(), (prediction["labels"][keep]-1).numpy(),
               prediction["scores"][keep].numpy(), ax=axes[1],
               title=f"Actual model output: {int(keep.sum())} boxes ≥ {CONFIDENCE}")
    plt.tight_layout(); plt.show()
    print("Raw output shapes:", {k: tuple(v.shape) for k, v in prediction.items()})
else:
    print("Run the optimizer demo to inspect this model's predictions.")


## 5. Exercises: move from a loop demonstration to an experiment

1. Explain why `num_classes=3` would be wrong for our three foreground classes.
2. Extend the loop to several epochs, evaluate the validation split after each
   epoch, and save the best state. Keep the test split untouched while tuning.
3. Compare YOLO and Faster R-CNN using the **same data split, resolution policy,
   device, warm-up, timing scope, and evaluation protocol**. Do not compare
   these two update steps with a fully trained YOLO checkpoint as a fair contest.
4. Verify that an empty target can use `boxes=torch.empty((0,4), dtype=float32)`
   and `labels=torch.empty((0,), dtype=int64)`. Why is `torch.tensor([])` insufficient
   for boxes?

<details><summary>Solutions / discussion</summary>

1. Background occupies ID 0; three foreground classes need 4 outputs.
2. Repeat the DataLoader loop, select a checkpoint using validation only, and
   evaluate all images with an established detection evaluator. Accumulating
   losses is useful for training diagnostics but is not mAP.
3. Report accuracy and latency separately. Warm up the model, synchronize CUDA
   when timing GPU work, and state whether preprocessing/postprocessing is timed.
4. `torch.tensor([])` has shape `(0,)`; the box contract requires `(N,4)` even
   when N=0. Labels also need the correct integer dtype.
</details>

**Troubleshooting:** An operator such as `torchvision::nms` missing usually means
incompatible PyTorch/TorchVision installations; reinstall a matched pair using
the project setup guidance. “Target labels out of bounds” → check 1–3 labels
and the four-class head. Out of memory → keep batch size 2, reduce image/proposal
sizes, or use CPU. Do not blindly lower to batch 1 with a trainable BatchNorm backbone.

**Checkpoint:** Show the target dictionary, identify background, explain each
optimizer step, and show the measured classifier weight change.

**Primary references:** [TorchVision detection fine-tuning tutorial](https://docs.pytorch.org/tutorials/intermediate/torchvision_tutorial.html),
[MobileNetV3 Faster R-CNN API](https://docs.pytorch.org/vision/stable/models/generated/torchvision.models.detection.fasterrcnn_mobilenet_v3_large_320_fpn.html),
[Faster R-CNN paper](https://arxiv.org/abs/1506.01497).
